In [ ]:
%pip install -e ../../../
%pip install plotly
%pip install nbformat

In [ ]:
import pandas as pd
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12816)

## Electricity Prices

Let's upload the electricity prices

In [ ]:
# Electricity price in NL
# from 2007-2025 half-yearly
# from https://ec.europa.eu/eurostat/databrowser/view/nrg_pc_205__custom_18701128/default/table
# in Euro per kWh
def get_electricity_price_data():
    with open('../raw-data/nrg_pc_205__custom_18701019_linear.csv', mode ='r')as file:
        csvFile = pd.read_csv(file)
    return np.array(csvFile['OBS_VALUE'].where(csvFile['tax']=='Excluding taxes and levies').dropna(), dtype=float)

prices = get_electricity_price_data()

start_date = '2007-01-01'
prices_df = pd.DataFrame(prices, columns=['NL0000'])
prices_df.index = pd.date_range(start=start_date, periods=len(prices), freq='6MS')

# Conservative extension for pre-2007 data
def extend_prices_conservative(prices, start_year=1990):
    """
    Extends electricity prices backwards using conservative assumptions.
    
    Parameters:
    - prices: array of prices from 2007 onwards (half-yearly)
    - start_year: year to start the extension from (default 1990)
    
    Returns:
    - DataFrame with extended prices and proper date index
    """
    # Calculate number of half-year periods from start_year to 2007
    periods_before = (2007 - start_year) * 2
    
    # Base price (first value from 2007)
    base_price = prices[0]
    
    # Generate pre-2007 prices with conservative assumptions
    extended_prices = []
    for i in range(periods_before - 1, -1, -1):
        # Trend factor: 85-95% of base price
        # Goes from 0.85 (furthest back) to 0.95 (closest to 2007)
        trend_factor = 0.85 + (i / periods_before) * 0.10
        
        # Small random noise: ±0.5 cents
        noise = (RNG.random() - 0.5) * 0.01
        
        # Calculate price
        price = base_price * trend_factor + noise
        extended_prices.append(max(0, price))  # Ensure non-negative
    
    # Combine extended prices with original prices
    all_prices = np.array(extended_prices + list(prices))
    
    # Create DataFrame
    df = pd.DataFrame(all_prices, columns=['NL0000'])
    
    # Create proper date index (half-yearly: Jan and Jul)
    start_date = f'{start_year}-01-01'
    df.index = pd.date_range(start=start_date, periods=len(all_prices), freq='6MS')
    df.index.name = 'timestamp'
    
    # Add a flag to indicate estimated vs actual data
    df['estimated'] = True
    df.loc[df.index >= '2007-01-01', 'estimated'] = False
    
    return df

# Actually create the extended DataFrame
prices_df = extend_prices_conservative(prices)

In [ ]:
import plotly.express as px 

fig = px.line(prices_df, x=prices_df.index, y='NL0000', title='Electricity Price Over Time')
fig.show()

## Electricity Price Pattern

Entso-e transparency paltfrom as the data from 2015. Before that we take the previous battle data

In [ ]:
# We represent the electiricty price pattern slightly differently: 1 value per column: multiplier of that hour of the week
price_patt_df = pd.DataFrame(columns=[f"NL0000-{i}" for i in range(0, 24*7)], index=pd.DatetimeIndex([], name='timestamp'))

ELE_PRICE_PATTERN_BWN2_WORKDAY = [6.72]*7+[10.94]*3+[27.68]*7+[10.94]*4+[6.72]*3
assert len(ELE_PRICE_PATTERN_BWN2_WORKDAY)==24
ELE_PRICE_PATTERN_BWN2_SATURDAY = [6.72]*7+[10.94]*13+[6.72]*4
assert len(ELE_PRICE_PATTERN_BWN2_SATURDAY)==24
ELE_PRICE_PATTERN_BWN2_SUNDAY = [6.72]*17+[10.94]*4+[6.72]*3
assert len(ELE_PRICE_PATTERN_BWN2_SUNDAY)==24
ELE_PRICE_PATTERN_BWN2 = ELE_PRICE_PATTERN_BWN2_WORKDAY*5 + ELE_PRICE_PATTERN_BWN2_SATURDAY + ELE_PRICE_PATTERN_BWN2_SUNDAY
assert len(ELE_PRICE_PATTERN_BWN2)==168

PPATT_2000 = np.array(ELE_PRICE_PATTERN_BWN2)
price_patt_df.loc[pd.to_datetime("2000-01-01"), :] = PPATT_2000/np.mean(PPATT_2000)

ELE_PRICE_PATT_2015_WINTER_WORKDAY = [1]*7+[3]*13+[1]*4
assert len(ELE_PRICE_PATT_2015_WINTER_WORKDAY) == 24
ELE_PRICE_PATT_2015_WINTER_WEEKEND = [1]*7+[2]*13+[1]*4
assert len(ELE_PRICE_PATT_2015_WINTER_WEEKEND) == 24
PPATT_2015_W = np.array(ELE_PRICE_PATT_2015_WINTER_WORKDAY*5+ELE_PRICE_PATT_2015_WINTER_WEEKEND*2)

ELE_PRICE_PATT_2015_SUMMER_WORKDAY = [0.9]*7+[1.9]*4+[1.1]*4+[2.6]*5+[0.9]*4
assert len(ELE_PRICE_PATT_2015_SUMMER_WORKDAY) == 24
ELE_PRICE_PATT_2015_SUMMER_WEEKEND = [0.9]*7+[1.5]*4+[1.1]*4+[2.1]*5+[0.9]*4
assert len(ELE_PRICE_PATT_2015_SUMMER_WEEKEND) == 24
PPATT_2015_S = np.array(ELE_PRICE_PATT_2015_SUMMER_WORKDAY*5+ELE_PRICE_PATT_2015_SUMMER_WEEKEND*2)

BASE = np.mean(np.concat([PPATT_2015_W,PPATT_2015_S]))

for y in range (2015, 2025):
    price_patt_df.loc[pd.to_datetime(f"{y}-04-01"), :] = PPATT_2015_S / BASE
    price_patt_df.loc[pd.to_datetime(f"{y}-10-01"), :] = PPATT_2015_W / BASE


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for idx, row in price_patt_df.iterrows():
    fig.add_trace(go.Scatter(
        x=list(range(len(row))),
        y=row.values,
        mode='lines',
        name=str(idx)
    ))

fig.update_layout(
    title="Electricity Price Pattern (Each Row)",
    xaxis_title="Hour of the Week",
    yaxis_title="Multiplier"
)
fig.show()

## Emission factor of the electricity Grid

We found cumulative data of greenhouses and production.

In [ ]:
# Greenhouse gases by electricity production in NL
# from 1990-2023
# from https://www.eea.europa.eu/en/analysis/maps-and-charts/greenhouse-gases-viewer-data-viewers
# in kt CO2 = 1.000.000 kg CO2
def get_greenhouse_gases_by_energy():
    with open('../raw-data/S_Table.csv', mode ='r', encoding='utf-16')as file:
        csvFile = pd.read_csv(file, sep="\t")
    arr =[]
    for x in np.array(csvFile['Emissions'].where((csvFile['Unnamed: 0']=="Netherlands") & (csvFile['Unnamed: 1']=="Energy supply")).dropna()):
        arr.append(float(x.replace('.','')))
    return arr

# Gross electricity production in NL
# from 1990-2023
# from https://ec.europa.eu/eurostat/databrowser/view/nrg_bal_c__custom_18717923/default/table
# in GWH = 1.000.000 KWH
def get_total_energy():
    with open('../raw-data/nrg_bal_c__custom_18717923_linear_2_0.csv', mode ='r')as file:
        csvFile = pd.read_csv(file)
    return np.array(csvFile['OBS_VALUE'], dtype=float)

# Greenhouse gases per energy production in NL
# from 1990-2023
# in kg CO2 per kWh
def get_greenhouse_gas_emissions_data():
    return get_greenhouse_gases_by_energy()/get_total_energy()

emission_factors = get_greenhouse_gas_emissions_data()

emis_f_df = pd.DataFrame(
    data=emission_factors,
    columns=['NL0000'],
    index=pd.date_range(
        start='1990-01-01',
        end='2023-01-01',
        freq='YS',
        name='timestamp'    
    )
)

# Add 2024 to 300 (trend interpolated by eye)
Y2024 = pd.to_datetime('2024-01-01')
emis_f_df.loc[Y2024] = 300

In [ ]:
fig = px.line(emis_f_df, x=emis_f_df.index, y='NL0000', title='Emission Factor Over Time')
fig.show()

## Solar panels costs

In [ ]:
# Unit cost for PV panel worldwide
# from 1975-2024
# from https://ourworldindata.org/grapher/solar-pv-prices
# in USD / watt = approx. Euro / kW *(1/1000)*(1/0.87)
def get_pv_panel_unit_cost():
    with open('../raw-data/solar-pv-prices.csv', mode ='r')as file:
        csvFile = pd.read_csv(file)
    return np.array(csvFile['Solar photovoltaic module price'], dtype=float) * 1000 * 0.87

panels_costs = get_pv_panel_unit_cost()

panels_costs_df = pd.DataFrame(
    data=panels_costs,
    columns=['NL0000'],
    index=pd.date_range(
        start='1975-01-01',
        end='2024-01-01',
        freq='YS',
        name='timestamp'    
    )
)

In [ ]:
fig = px.line(panels_costs_df, x=panels_costs_df.index, y='NL0000', title='Cost of Solar Panel Over Time')
fig.show()

In [ ]:
from water_futures_battle.core.base_model import DynamicProperties
energy_dps = DynamicProperties('energy_system-dynamic_properties')

from water_futures_battle.energy.dynamic_properties import EnergySysDB

mask = (prices_df.index >= pd.to_datetime("2000-01-01")) & (prices_df.index < pd.to_datetime("2025-01-01"))
energy_dps[EnergySysDB.EPRICE_UNIT] = prices_df.loc[mask, 'NL0000'].copy()

energy_dps[EnergySysDB.EPRICE_PATT] = price_patt_df.copy()

mask = (emis_f_df.index >= pd.to_datetime("2000-01-01")) & (emis_f_df.index < pd.to_datetime("2025-01-01"))
energy_dps[EnergySysDB.EMISS_FACTOR] = emis_f_df.loc[mask, 'NL0000'].copy()

mask = (panels_costs_df.index >= pd.to_datetime("2000-01-01")) & (panels_costs_df.index < pd.to_datetime("2025-01-01"))
energy_dps[EnergySysDB.SOLAR_COST] = panels_costs_df.loc[mask, 'NL0000']

energy_dps.dump('..')